## The first big project - The Digital Twin

### But first: introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like Agents) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

## Heads up - a change from the videos

In the video, I deploy the twin for free to HuggingFace Spaces. HuggingFace has recently stopped supporting this for free!

There is a free alternative, and I explain it and give instructions later on in this lab.

In [1]:
# WHAT THIS CELL DOES:
# Bring in the libraries (ready-made code) this notebook needs.
# dotenv reads the .env file, anthropic talks to Claude, json and os are Python basics,
# requests sends web requests, PdfReader reads PDF files, gradio builds the chat window.

# imports

from dotenv import load_dotenv
import anthropic
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [2]:
# WHAT THIS CELL DOES:
# Load the keys from .env and create the connection to Claude.
# MODEL is the Claude model we use everywhere below.
# get_text() pulls the plain text out of Claude's answer.

# The usual start

load_dotenv(override=True)
claude = anthropic.Anthropic()

MODEL = "claude-opus-5-5"

# Claude returns a list of content blocks. This joins the text blocks into one string.
def get_text(response):
    return "".join(block.text for block in response.content if block.type == "text")

In [3]:
# WHAT THIS CELL DOES:
# Read the two Pushover keys from .env and check that they look right.
# Pushover is the service that sends notifications to your phone.

# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

Pushover user found and looks good
Pushover token found and looks good


In [4]:
# WHAT THIS CELL DOES:
# A function that sends one notification to your phone.
# It prints the message, then posts it to Pushover together with your two keys.

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [7]:
# WHAT THIS CELL DOES:
# Try it out: this sends a real "HEY!!" notification to your phone.

push("HEY!!")

Push: HEY!!


In [10]:
# WHAT THIS CELL DOES:
# Tool 1: when a visitor leaves their email, send yourself a notification with their details.
# name and notes are optional. If Claude does not give them, the default text is used.
# It returns "OK" so Claude knows the tool worked.

def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [11]:
# WHAT THIS CELL DOES:
# Tool 2: when the twin cannot answer a question, send yourself a notification with that question.
# This way you learn what visitors ask that your profile does not cover.

def record_unknown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return "OK"

In [12]:
# WHAT THIS CELL DOES:
# Describe tool 1 to Claude. Claude cannot see the Python function, so we explain it here:
# its name, when to use it, and which inputs it takes (email is required, name and notes are optional).

record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "input_schema": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"},
            "name": {"type": "string", "description": "The user's name, if they provided it"},
            "notes": {"type": "string", "description": "Any additional info about the conversation that's worth recording to give context"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [13]:
# WHAT THIS CELL DOES:
# Describe tool 2 to Claude in the same way: its name, when to use it, and its one input (the question).

record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
    "input_schema": {
        "type": "object",
        "properties": {
            "question": {"type": "string", "description": "The question that couldn't be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [14]:
# WHAT THIS CELL DOES:
# Put both tool descriptions in one list. We hand this list to Claude with every request.

# Claude takes the tool descriptions directly, with no extra wrapper

tools = [record_user_details_json, record_unknown_question_json]

In [15]:
# WHAT THIS CELL DOES:
# Show the list, just to look at it.

tools

[{'name': 'record_user_details',
  'description': 'Use this tool to record that a user is interested in being in touch and provided an email address',
  'input_schema': {'type': 'object',
   'properties': {'email': {'type': 'string',
     'description': 'The email address of this user'},
    'name': {'type': 'string',
     'description': "The user's name, if they provided it"},
    'notes': {'type': 'string',
     'description': "Any additional info about the conversation that's worth recording to give context"}},
   'required': ['email'],
   'additionalProperties': False}},
 {'name': 'record_unknown_question',
  'description': "Always use this tool to record any question that couldn't be answered as you didn't know the answer",
  'input_schema': {'type': 'object',
   'properties': {'question': {'type': 'string',
     'description': "The question that couldn't be answered"}},
   'required': ['question'],
   'additionalProperties': False}}]

In [16]:
# WHAT THIS CELL DOES:
# Run the tools Claude asked for (version 1, with an if statement).
# Claude only ASKS for a tool. Our code has to actually run it.
# For each request: read the tool name and inputs, call the matching Python function,
# and pack the result in the format Claude expects back (a "tool_result").

# This function can take a list of tool calls, and run them. This is the IF statement!!
# With Claude, each tool call is a "tool_use" block: block.name is the tool, block.input holds the arguments (already a dict)

def handle_tool_calls_with_manual_if(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.name
        arguments = tool_call.input
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        else:
            result = "No tool found"

        results.append({"type": "tool_result", "tool_use_id": tool_call.id, "content": json.dumps(result)})
    return results

## Using Python built-in globals()

Python has a dictionary that gives us access to all global functions.

Sidenote: for sure when we deploy, we will use this in a more protected way..

In [17]:
# WHAT THIS CELL DOES:
# A Python trick: globals() is a dictionary of everything defined in this notebook.
# So globals()["record_unknown_question"] finds the function by its name as text, and (...) calls it.
# Careful: this sends a real notification to your phone.

globals()["record_unknown_question"]("this is a really hard question")

Push: Recording this is a really hard question asked that I couldn't answer


'OK'

In [18]:
# WHAT THIS CELL DOES:
# Run the tools Claude asked for (version 2, without the if statement).
# Instead of one if per tool, it looks the function up by name with globals().
# Add a new tool later and this code still works without changes.

# This gives us a more elegant way that avoids the IF statement.

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.name
        arguments = tool_call.input
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({"type": "tool_result", "tool_use_id": tool_call.id, "content": json.dumps(result)})
    return results

In [19]:
# WHAT THIS CELL DOES:
# Read your LinkedIn PDF page by page into one long text called linkedin.
# Then read your summary.txt into a text called summary.

reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [20]:
# WHAT THIS CELL DOES:
# Build the system prompt: the instructions that tell Claude who it is and how to behave.
# The f at the start lets us insert {summary} and {linkedin} into the text.
# The rules at the end tell Claude when to use the two tools.

system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions related to career, background, skills and experience.
If the user asks about something unrelated, then steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

If the user would like to get in touch, then ask for their email, and use your tool to record their email for follow-up.

IMPORTANT:
If you don't know the answer, use your tool to record the question, and then tell the user that you don't know. Never make up an answer.
"""


In [21]:
# WHAT THIS CELL DOES:
# The heart of the app. chat() runs every time a visitor sends a message.
# 1. clean_history() tidies the old messages so Claude accepts them.
# 2. Send the conversation, the system prompt and the tools to Claude.
# 3. While Claude asks for a tool: run it, send the result back, and ask Claude again.
# 4. When Claude stops asking for tools, return its text answer.

# Gradio's history entries carry extra fields that Claude's API rejects, so keep only role and text
def clean_history(history):
    cleaned = []
    for h in history:
        content = h["content"]
        if isinstance(content, list):
            content = "".join(part.get("text", "") for part in content if isinstance(part, dict))
        if isinstance(content, str) and content:
            cleaned.append({"role": h["role"], "content": content})
    return cleaned

def chat(message, history):
    messages = clean_history(history) + [{"role": "user", "content": message}]
    response = claude.messages.create(model=MODEL, max_tokens=4096, system=system_prompt, messages=messages, tools=tools)
    while response.stop_reason == "tool_use":
        tool_calls = [block for block in response.content if block.type == "tool_use"]
        results = handle_tool_calls(tool_calls)
        messages.append({"role": "assistant", "content": response.content})
        messages.append({"role": "user", "content": results})
        response = claude.messages.create(model=MODEL, max_tokens=4096, system=system_prompt, messages=messages, tools=tools)
    return get_text(response)

In [22]:
# WHAT THIS CELL DOES:
# Open a chat window in your browser. Every message typed there goes through chat().

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarn

Tool called: record_unknown_question
Push: Recording What is your favorite developer in the tech industry? asked that I couldn't answer


/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: record_user_details
Push: Recording interest from Name not provided with email testme@email.com and notes Visitor asked about Amine's skills, then asked about his favorite actor (off-topic) and his favorite developer in the tech industry (unknown). They then asked to get in touch. No name given.
Tool called: record_unknown_question
Push: Recording Who is your favorite developer in the tech industry? asked that I couldn't answer


/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Tool called: record_user_details
Push: Recording interest from Jason with email testme@email.com and notes Jason wants to get in touch. He asked about Amine's skills, his favorite actor (off-topic, not answered) and his favorite developer in the tech industry (unknown, recorded separately). He has not said yet what he'd like to discuss.
Tool called: record_unknown_question
Push: Recording Who is your favorite developer in the tech industry? asked that I couldn't answer


/Users/aminejib/Desktop/BW/Arbeit_Bewerbung/agents/.venv/lib/python3.12/site-packages/gradio/routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


## Turning into Python modules

I've turned the code in the lab into python modules; that's a great practice to do after you've completed experiments in the Notebook.

You could put all the code above into 1 python script. But it's nicer to organize the code into different modules for different concerns, and that's what I've done:

`context.py` loads in the static data and constructs the System Prompt

`tools.py` contains all the code to manage and call tools, with their associated json

`app.py` contains the Gradio app and OpenAI call.

`styles.py` contains styles to apply to Gradio and this was entirely written by Claude Code!

You could have a stab at doing this yourself, then compare with my versions.

Then to try it out, open a terminal in Cursor:

`cd 1_foundations`  
`cd twin`  
`uv run app.py`

# STOP THE PRESS! Heads up...

As of 9-July-2026, HuggingFace has suddenly stopped allowing Gradio Apps to be deployed for free on HuggingFace Spaces.

This is quite a nasty surprise!

I expect they might reverse this decision. In the meantime, here's a free alternative: using Render.

You'll find complete instructions in [the file RENDER_INSTRUCTIONS in this directory](RENDER_INSTRUCTIONS.md)

If you don't mind paying for HuggingFace, the original instructions are below.

And also, here are instructions on my digital twin, which runs at very low cost on fly.io:  
https://edwarddonner.com/avatar

With my twin, not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg

## Original Instructions with HF Spaces (no longer free)

We will deploy to HuggingFace Spaces.

Before you start: remember to update the files in the `twin` directory - your LinkedIn profile and summary.txt - so that it talks about you!

Also check that there's no README file within the twin directory. If there is one, please delete it. The deploy process creates a new README file in this directory for you.

## Deployment Part 1: HuggingFace

1. Visit https://huggingface.co and set up an account  
2. From the Avatar menu on the top right, choose Access Tokens. Choose "Create New Token". Give it WRITE permissions - it needs to have WRITE permissions! Keep a record of your new key.  
3. In the Cursor Terminal, run: `uvx hf auth login --token YOUR_TOKEN_HERE`, like `uvx hf auth login --token hf_xxxxxx`, to login at the command line with your key. Afterwards, run `uvx hf auth whoami` to check you're logged in  
4. Take your new token and add it to your .env file: `HF_TOKEN=hf_xxx` for the future

## Deployment Part 2: Push!

1. Go in to the twin directory: `cd 1_foundations` then `cd twin`
2. From the twin directory, enter: `uv run gradio deploy` 
3. Follow its instructions by selecting the default values: name it `twin`, specify app.py, choose cpu-basic as the hardware, say No to needing to supply secrets, and say "no" to github actions.  

### Deployment Part 3: Secrets

1. Go to https://huggingface.co and click your Avatar, go to your profile, select the Space
2. Go to the 3 dots menu and pick Settings
3. Scroll down to Variables and Secrets section
4. Press "New Secret" (not New Variable) and enter the name of `OPENAI_API_KEY` and the value of your key from the .env file (or use the relevant key for your LLM). Be careful to get this right!
5. Repeat for `PUSHOVER_USER` and `PUSHOVER_TOKEN` from your .env file
6. Nearer the top of the settings, click "Restart space" to restart it
7. Click on App near the top to return to the app, and after it has restarted - enjoy!

### Embedding in another site

To embed this in another website, select "Embed this space" from the three-dots menu.

### Troubleshooting

If you get a gradio error, try opening the logs (the button next to the 3-dot menu).  
Try adding more debug information particularly around your keys.

### Redploying the space

Just run `uv run gradio deploy` from the twin directory. You might need to delete the file README.md that Gradio created there if you want to name your space again.

### Deleting the space

From the 3 dots menu, select the Settings screen, and there's a Delete option at the bottom.

For more information on deployment:

https://www.gradio.app/guides/sharing-your-app#hosting-on-hf-spaces

### My Digital Twin

So I spend some time taking my Digital twin to the next level!  
Here it is:   
https://edwarddonner.com/avatar

Not only can you notify me with a Push, but you can chat with the real me! Here's a video with how I made it, and instructions if you want to make it too. I started with this Career Conversations app.  
https://youtu.be/srlhW4H-Gtg


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">• First and foremost, deploy this for yourself! It's a real, valuable tool - the future resume..<br/>
            • Next, improve the resources - add better context about yourself. If you know RAG, then add a knowledge base about you.<br/>
            • Add in more tools! You could have a SQL database with common Q&A that the LLM could read and write from?<br/>
            • Bring in the Evaluator from the exercise in Day 4, and add other Agentic patterns.<br/>
            • Some students have added Telegram integration so that you can chat live with people on your site, along with your twin!
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">Aside from the obvious (your resume of the future) this has business applications in any situation where you need an AI assistant with domain expertise and an ability to interact with the real world.
            </span>
        </td>
    </tr>
</table>